In [1]:
# Mod 1 Lesson 12 (copied from lesson 8, with minor edit to instructions/rag assistant)

# After completing your 'ingest.py' and 'rag_helper.py' files
# Import from those files and put it all together:

# Load environmental variables from .env file (includes llm key)
from dotenv import load_dotenv
load_dotenv()

# Import Anthropic client
# Initialize Anthropic client (picks API key from .env)
from anthropic import Anthropic 
anthropic_client = Anthropic()


# Import data loading and index building functions from ingest.py
from ingest import load_faq_data, build_index

# Fetch all FAQ documents from DataTalks.Club and build the search index
documents = load_faq_data() #already has the URLs and loads from FAQs
index = build_index(documents) #refers to index function created in 'ingest.py'


# Import RAG class from rag_helper.py
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

# Create RAG assistant, passing in the index and LLM client
assistant = RAGBase(
    index=index,
    llm_client=anthropic_client,
    instructions=instructions,
)

In [2]:
# Test limitations of this model using correct & incorrect spellings

answer = assistant.rag("How do I run Ollama locally?")
print(answer)

print("------")

answer = assistant.rag("How do I run Olama locally?")
print(answer)

# How to Run Ollama Locally

Based on the course materials, here are the steps to run Ollama locally:

## 1. Installation
First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choose your operating system:

- **macOS**: Download the `.pkg` and install it
- **Windows**: Download the `.msi` and install it
- **Linux**: Run this command in the terminal:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

## 2. Running a Model
Once installed, open a terminal and run:
```bash
ollama run llama3
```

This will:
- Download the LLaMA 3 model (~4GB)
- Start the model locally
- Open a chat-like interface where you can type questions

## 3. Testing the Server
To verify the Ollama local server is running:
```bash
curl http://localhost:11434
```

You should receive a response with model information.

## 4. Using with Python
Install the Python client:
```bash
pip install ollama
```

Then use this minimal example:
```python
import ollama

response

In [3]:
# Mod 1 Lesson 13 - Asking Without Tools

# First see what LLM does without tools (does not have context; just uses its general knowledge)

messages = [
   {"role": "user", 
    "content": "I just discovered the course. Can I join it?"}
]

response = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=messages,
    )

print(response.content[0].text)

I'd be happy to help, but I need a bit more information! Could you clarify:

1. **Which course** are you interested in?
2. **Where did you discover it?** (a website, institution, platform, etc.)
3. **When does it start?**

With those details, I can give you better information about enrollment, deadlines, prerequisites, or how to join.


In [4]:
# Defining the tool 
# 1. define top-level 'search' function that queries the 'index' directly

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict= {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [5]:
# 2. tell the model about the function
# the llm reads the 'description' to decide when to call the function
# 'input_schema' ('parameters' in OpenAI) is the JSON schema for the arguments
# mark 'query' as required so the model always fills it in

search_tool = {
    #"type": "function", <-- Anthropic doesn't use this
    "name": "search",
    "description": "Search the FAQ for entries matching the given query",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ"
            }
        },
        "required": ["query"]
        #"additionalProperties": False <-- Anthropic doesn't use this
    }
}

In [6]:
# Send the question with the tool this time:

response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=messages,
    tools=[search_tool],
)

print(response.content[0].text)

I'll search the FAQ for information about joining the course.


In [7]:
# 3. Execute the function (contains JSON arguments)
# parse them, then call the 'search' function, and serialize the results

# extract the tool_use block from the response 
# Anthropic uses stop_reason="tool_use" instead of output[0].type="function_call" 

# import JSON library for converting Python objects to/from JSON strings
import json 

# loop through all blocks in the response (response.content is a list of blocks) and find the first one where type == "tool_use". This is the tool call the model made. 'next()' grabs the first match and stops.
call = next(b for b in response.content if b.type == "tool_use")

# extract the arguments from the tool call. The model provided "query": "join course enrollment" which is already a Python dict, so you don't need to parse the JSON that OpenAI returns. 
args = call.input # don't need json.loads() like OpenAI

# execute the saerch function with the model's rewritten query
# 'search(**{"query": "join course enrollment"})' becomes 'search(query="join course enrollment"). ** unpacks the dict into named parameters. 
results = search(**args)

# converts the search results (Python dicts) into formatted JSON string with 2-space indentation. This is what you'll send back to the model as the tool result. 
result_json = json.dumps(results, indent=2)

In [8]:
# see the rewritten query in the call:
print(call)

ToolUseBlock(id='toolu_01JqF2CEJGwdWTtALNTnLz2c', caller=DirectCaller(type='direct'), input={'query': 'how to join the course enroll'}, name='search', type='tool_use')


In [12]:
# 4. Now send this result back to the model
# first, add the model's output to the conversation history
# the model needs to see its own function call
# then add the tool result

# Append assistant's full response to messages (Anthropic requires the full content list)
messages.append({"role": "assistant", "content": response.content})

# Append the tool result as a user message with tool_result type
# Anthropic wraps tool results in a user message, unlike OpenAI's "function_call_output" type
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": call.id, # instead of OpenAI's "call_id" 
        "content": result_json
    }]
})

In [14]:
# Now ask the model again. This time with expanded history:

response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=messages,
    tools=[search_tool]
)

print(response.content[0].text)

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.3: `tool_use` ids were found without `tool_result` blocks immediately after: toolu_014J5DUZx9hcCDPcyDH7MpdM. Each `tool_use` block must have a corresponding `tool_result` block in the next message.'}, 'request_id': 'req_011CcHbH3RH2aexSaA7cGWrf'}

In [ ]:
# The model above has the original question, its own decision to call 'search', and the FAQ results. It can now produce a course-specific answer. 

# With plain RAG we made one call, and with "agentic RAG" (or "tool use" or "function calling") we make two calls. This is a full function-calling loop for a single turn. The LLM decides which tools to call.

In [15]:
# TOKEN USAGE AND COST
# Each call you make costs money for the model

usage = response.usage
usage.input_tokens, usage.output_tokens

(584, 108)

In [16]:
# For each model, the provider publishes a price per million input tokens and per million output tokens. Plug these in to convert tokens to dollars:

def calc_claude_h45_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 1.00
    OUTPUT_PRICE_PER_MILLION = 5.00

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost 

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calc_claude_h45_price(1433, 324)
print("Total cost: $", round(result["total_cost"], 8))

print("----")

result = calc_claude_h45_price(usage.input_tokens, usage.output_tokens)
print("Input tokens:", usage.input_tokens, ", Output tokens:", usage.output_tokens, ", Total cost: $", round(result["total_cost"],8))

Total cost: $ 0.003053
----
Input tokens: 584 , Output tokens: 108 , Total cost: $ 0.001124


In [17]:
# LESSON 14: Agentic Loop
# Now we want the llm to decide how many calls to make 

# Write instructiosn to give the agent its role:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [18]:
# A function-call helper
# We will run function calls (tool use) repeatedly inside a loop ,so wrap that in a small helper. This turns the JSON arguments into a Python dict, calls the right function, and serializes the result. We only have one tool for now, so we dispatch on the function name directly:

def make_call(tool_use):
    # Extract the arguments from the tool_use block
    # Anthropic already gives us a dict, no json.loads() needed like OpenAI
    args = tool_use.input

    # Check which tool was called and execute it
    if tool_use.name == "search":
        result = search(**args)

    # Convert the search results to a JSON string for the response
    result_json = json.dumps(result, indent=2)

    # Return the tool result in Anthropic's format
    # Note: This gets wrapped in a user message with role="user" when appending to messages
    return {
        "type": "tool_result",           # Anthropic uses "tool_result" instead of "function_call_output"
        "tool_use_id": tool_use.id,      # Anthropic uses "tool_use_id" instead of "call_id"
        "content": result_json,          # Anthropic uses "content" instead of "output"
    }

# The helper returns the exact structure the Responses API expects.

In [19]:
# PROCESSING ONE RESPONSE
# Process a single model response. Append each output entry to the conversation, print any messages, and run any function calls (tool use). Tool use results get appended too. 

# Question to send:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "user", "content": question},
]

# First call — send question with tool available
response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    #system=instructions,
    messages=messages,
    tools=[search_tool],
)

# Convert Block objects to dicts and append assistant response to history
messages.append({
    "role": "assistant",
    "content": serialize_content(response.content)
})

has_tool_calls = response.stop_reason == "tool_use"

if has_tool_calls:
    # Get all tool_use blocks and execute them
    tool_uses = [b for b in response.content if b.type == "tool_use"]
    tool_results = []

    for tool_use in tool_uses:
        print("tool_use:", tool_use.name, tool_use.input)
        tool_result = make_call(tool_use)
        tool_results.append(tool_result)

    # Append all tool results in one user message
    messages.append({
        "role": "user",
        "content": tool_results
    })

elif response.stop_reason == "end_turn":
    print("ASSISTANT:")
    print(response.content[0].text)

NameError: name 'serialize_content' is not defined

In [ ]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'd be happy to help you find information about joining the course! Let me search for enrollment and admission details."},
   {'type': 'tool_use',
    'id': 'toolu_01113ehC3tThcp5KMkPzzg8c',
    'name': 'search',
    'input': {'query': 'join course enrollment admission'}},
   {'type': 'tool_use',
    'id': 'toolu_01SvVmm8SysBVkJYXSMd4KYA',
    'name': 'search',
    'input': {'query': 'how to register sign up'}},
   {'type': 'tool_use',
    'id': 'toolu_01Mh3jmKiEY5w5GRQ2AojRRa',
    'name': 'search',
    'input': {'query': 'course requirements prerequisites'}}]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01113ehC3tThcp5KMkPzzg8c',
    'content': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the cours

In [ ]:
# Reset messages — clears any broken state from previous runs
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "user", "content": question}
]

it = 1

while True:
    print(f"iteration #{it}...")

    response = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system=instructions,
        messages=messages,
        tools=[search_tool],
    )

    messages.append({
        "role": "assistant",
        "content": serialize_content(response.content)
    })

    if response.stop_reason == "tool_use":
        tool_uses = [b for b in response.content if b.type == "tool_use"]
        tool_results = []

        for tool_use in tool_uses:
            print("tool_use:", tool_use.name, tool_use.input)
            tool_result = make_call(tool_use)
            tool_results.append(tool_result)

        messages.append({
            "role": "user",
            "content": tool_results
        })

    elif response.stop_reason == "end_turn":
        print("ASSISTANT:")
        print(response.content[0].text)
        break

    it += 1

iteration #1...
tool_use: search {'query': 'join enroll course registration'}
tool_use: search {'query': 'course enrollment deadline when can I join'}
iteration #2...
ASSISTANT:
Great news! **Yes, you can join the course!** Here's what you need to know:

**Key Points:**
- ✅ **You can start learning anytime** - the videos and materials are available
- ✅ **You don't need formal registration approval** - you can just start learning and submitting homework (while the form is open) without being checked against a registered list
- ✅ **Registration is optional** - it's just to gauge interest before the official start date

**Important for Certificates:**
If you want to receive a **certificate**, you need to:
- Submit your project **while submissions are still being accepted** (there's a deadline for this)
- Note that certificates are only awarded if you finish with a "live" cohort, not in self-paced mode (because you'll need to peer-review 3 capstone projects after submitting yours)

**How t